# 25. Metadata Catalogs

*Module: 6 — Python access · Companion summary: `lectures/25-metadata-catalogs.md` · SQL engine: SQLite (in-memory)*

## What you will study

- ask the database *about itself* with catalog queries
- browse sqlite_master (SQLite's system catalog)
- get column-level metadata with PRAGMA table_info

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Build something to catalog — the course AUTHOR table

In [ ]:
q("CREATE TABLE author (author_id TEXT PRIMARY KEY, lastname TEXT NOT NULL, firstname TEXT NOT NULL, email TEXT, city TEXT, country TEXT)")
q("INSERT INTO author VALUES ('A1', 'Chong', 'Raul', 'RFC@ca.ibm.com', 'Toronto', 'CA')")
q("CREATE TABLE book (book_id TEXT PRIMARY KEY, title TEXT, author_id TEXT REFERENCES author(author_id))")
q("INSERT INTO book VALUES ('B1', 'My Book', 'A1')")

## 🧪 sqlite_master — the list of every object in the database

In [ ]:
q("SELECT name, type FROM sqlite_master")

## 🧪 The full CREATE statement, as recorded in the catalog

In [ ]:
q("SELECT sqlite_master.sql FROM sqlite_master WHERE name = 'author'")

## 🧪 PRAGMA table_info — the column report

In [ ]:
q("PRAGMA table_info(author)")

**Reading the columns**: `cid, name, type, notnull, dflt_value, pk` —
`author_id` has `pk: 1`; lastname/firstname have `notnull: 1`, matching the DDL.

In [ ]:
q("PRAGMA table_info(book)")

## 💡 Tips & tricks

- Catalog queries are how tools and ORMs introspect a schema — and how *you* write
  schema-checking scripts (does `email` exist before migrating? now you can ask).
- 📕 Dialect note: DB2/Oracle call these SYSTEM CATALOG / SYSCAT / SYSIBM tables
  (`SYSCAT.TABLES`), MySQL has `information_schema.tables`; SQLite's lightweight
  equivalent is `sqlite_master` + PRAGMAs.
- The catalog is a *read-mostly* table: never hand-edit it; use DDL.

## 🎯 Your turn

1. List every table (type='table') excluding the ones whose name starts with
   `sqlite_` (internal).
2. Write a tiny introspector: for each user table, print its name and column count.

In [ ]:
# 💡 Solution
q("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'")
for (tname,) in conn.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'"):
    n = conn.execute(f"PRAGMA table_info({tname})").fetchall()
    print(f'{tname}: {len(n)} columns')